# CYBERMIND — Colab T4 Final Training

**Goal:** use a free Colab T4 as the training machine, train **one compact CYBERMIND model**, save the real trained `.pt` artifact, export ONNX, verify it, and download the final package.

The notebook is intentionally practical: **Colab provides the GPU; the repo scripts perform the actual training.** Gemini/Colab agent can inspect and edit the repo files in this runtime.


In [ ]:
!nvidia-smi
import torch, os, sys
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))


## 1) Upload the CYBERMIND package

Upload the ZIP produced for this project. This avoids requiring the repo to already be public on GitHub.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.endswith('.zip')), None)
if not zip_name:
    raise RuntimeError("Please upload the CYBERMIND .zip package.")
print("Uploaded:", zip_name)


In [ ]:
import os, zipfile, shutil, pathlib
ROOT = pathlib.Path('/content/CYBERMIND')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile('/content/' + zip_name) as z:
    z.extractall(ROOT)
# Handle a single top-level folder inside the archive.
children = list(ROOT.iterdir())
if len(children) == 1 and children[0].is_dir() and (children[0] / 'scripts').exists():
    REAL = children[0]
else:
    REAL = ROOT
print('Project:', REAL)
os.chdir(REAL)


## 2) Bootstrap the runtime

Do not reinstall the Colab CUDA/PyTorch stack unnecessarily. Install the project dependencies and the graph/runtime packages only when needed.


In [ ]:
%cd $REAL
!python -m pip install -q pyyaml tqdm pandas scikit-learn pyarrow networkx matplotlib joblib onnx onnxruntime onnxscript awscli
!python -m pip install -q torch-geometric
!python scripts/gpu_preflight.py --config configs/colab_t4.yaml


## 3) Optional: mount Google Drive for persistent checkpoints

This is strongly recommended for long runs. Training data remains on the Colab VM for speed; important artifacts are copied to Drive after training.


In [ ]:
USE_DRIVE = True
DRIVE_OUT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/CYBERMIND_FINAL'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('Persistent output folder:', DRIVE_OUT)
else:
    print('Drive persistence disabled.')


## 3b) Download public datasets & knowledge bases (all no-login, direct)

Every source below is confirmed publicly downloadable with no account: CIC-IDS2018 (AWS S3), CIC-IDS2017, CICIoT2023, CTU-13, UNSW-NB15 (HuggingFace mirror), MITRE ATT&CK, CAPEC, and NVD/CVE. UNSW-NB15's official UNSW host and the DARPA 1998/99 per-week files are manual-only (no single scriptable URL) — see `data/manifests/DOWNLOAD_MATRIX.md` if you want those too. CIC-IDS2018 alone is already pulled again in the training step below; run this cell first for the rest.


In [ ]:
!python scripts/download_public_sources.py \
  --source cic2017 --source ciciot2023 --source ctu13 --source unsw_nb15_hf \
  --source attack_enterprise --source capec_latest --source nvd_cve_recent


## 4) One real-data training run

This does: dataset acquisition → normalization → strict validation → graph preparation → baseline → training → evaluation → final `.pt` creation → ONNX export/verification.

The T4 profile uses the same compact architecture intended for the 8 GB RTX 5050/5060 laptops.


In [ ]:
# Full production-style run
!python scripts/one_click_train.py --config configs/colab_t4.yaml


## 5) Resume after a disconnect

When a checkpoint is available, use the same command with `--resume`. If the Colab VM was fully reset, restore the project/checkpoint from Drive first.


In [ ]:
# Example:
# !python scripts/one_click_train.py --config configs/colab_t4.yaml --resume checkpoints/cybermind_compact.pt


## 6) Inspect the final artifacts


In [ ]:
from pathlib import Path
for p in [Path('checkpoints/cybermind_compact.pt'), Path('models/cybermind_final.pt'), Path('export/cybermind_compact.onnx')]:
    print(p, '->', f'{p.stat().st_size/1024/1024:.2f} MB' if p.exists() else 'MISSING')
!find results -maxdepth 2 -type f | sort | tail -80


## 7) Copy the final package to Google Drive and download it


In [ ]:
import shutil, os
if DRIVE_OUT:
    for src in ['models/cybermind_final.pt','models/cybermind_final.json','export/cybermind_compact.onnx','checkpoints/cybermind_compact.pt','results/train_history.json']:
        s=Path(src)
        if s.exists():
            dst=Path(DRIVE_OUT)/s.name
            shutil.copy2(s,dst)
            print('Drive:', dst)

final_zip = Path('/content/CYBERMIND_FINAL_ARTIFACTS.zip')
base = Path('/content/CYBERMIND_FINAL_ARTIFACTS')
if base.exists(): shutil.rmtree(base)
base.mkdir()
for src in ['models/cybermind_final.pt','models/cybermind_final.json','export/cybermind_compact.onnx','results/train_history.json']:
    s=Path(src)
    if s.exists(): shutil.copy2(s, base/s.name)
shutil.make_archive(str(final_zip.with_suffix('')), 'zip', base)
print('FINAL PACKAGE:', final_zip)


In [ ]:
from google.colab import files
if Path('/content/CYBERMIND_FINAL_ARTIFACTS.zip').exists():
    files.download('/content/CYBERMIND_FINAL_ARTIFACTS.zip')


## Gemini / Colab agent workflow

Use the built-in Gemini/agent UI to inspect or patch files in `/content/CYBERMIND`. Give it the `AGENT.md` instructions and ask it to:

> inspect the current failure, patch the minimal files, run the smallest relevant test, and do not delete the existing checkpoint.

After a successful patch, rerun the affected notebook cell or the one-click command.
